In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
# Cella 1 - installa pacchetti e scarica SOLO i dati (niente UMAP qui)
!pip install -q umap-learn huggingface_hub

import numpy as np
from huggingface_hub import hf_hub_download

REPO_ID = "R-obi/ai-text-detection"
REPO_TYPE = "dataset"

emb_path = hf_hub_download(repo_id=REPO_ID, repo_type=REPO_TYPE,
                            filename="train/train_embeddings.npy")
lbl_path = hf_hub_download(repo_id=REPO_ID, repo_type=REPO_TYPE,
                            filename="train/train_labels.npy")

embeddings = np.load(emb_path)
labels = np.load(lbl_path)

print("Embeddings shape:", embeddings.shape)
print("Labels shape:", labels.shape)

train/train_embeddings.npy:   0%|          | 0.00/2.36G [00:00<?, ?B/s]

train/train_labels.npy:   0%|          | 0.00/4.62M [00:00<?, ?B/s]

Embeddings shape: (577300, 1024)
Labels shape: (577300,)


In [3]:
# Cella 2 - UMAP (memoria minima) + 3 grafici densita'
import gc
import numpy as np
import plotly.graph_objects as go
from sklearn.decomposition import PCA
import umap

embeddings = embeddings.astype(np.float32)

N_MAX = 60_000
if embeddings.shape[0] > N_MAX:
    rng = np.random.default_rng(42)
    idx_sample = rng.choice(embeddings.shape[0], N_MAX, replace=False)
    emb_sub = embeddings[idx_sample]
    lbl_sub = labels[idx_sample]
else:
    idx_sample = np.arange(embeddings.shape[0])
    emb_sub = embeddings
    lbl_sub = labels

del embeddings, labels
gc.collect()

pca = PCA(n_components=50, random_state=42)
emb_pca = pca.fit_transform(emb_sub)
del emb_sub
gc.collect()

reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    n_components=2,
    n_jobs=-1,
    verbose=True,
    low_memory=True,
)
emb_2d = reducer.fit_transform(emb_pca)
del emb_pca
gc.collect()

x, y = emb_2d[:, 0], emb_2d[:, 1]
hover_text = [f"idx: {i}<br>label: {l}" for i, l in zip(idx_sample, lbl_sub)]

mask1 = lbl_sub == 1
mask0 = lbl_sub == 0

# --- Grafico 1: densita' label 1 ---
fig1 = go.Figure()
fig1.add_trace(go.Histogram2dContour(
    x=x[mask1], y=y[mask1],
    colorscale="Reds",
    ncontours=25,
    contours=dict(coloring="heatmap"),
))
fig1.update_layout(title="Densita' - Label 1", xaxis_title="UMAP 1",
                    yaxis_title="UMAP 2", width=800, height=650)
fig1.show()
fig1.write_html("umap_density_label1.html")

# --- Grafico 2: densita' label 0 ---
fig0 = go.Figure()
fig0.add_trace(go.Histogram2dContour(
    x=x[mask0], y=y[mask0],
    colorscale="Blues",
    ncontours=25,
    contours=dict(coloring="heatmap"),
))
fig0.update_layout(title="Densita' - Label 0", xaxis_title="UMAP 1",
                    yaxis_title="UMAP 2", width=800, height=650)
fig0.show()
fig0.write_html("umap_density_label0.html")

# --- Grafico 3: solo curve di livello, entrambe le label sovrapposte ---
fig_lines = go.Figure()
fig_lines.add_trace(go.Histogram2dContour(
    x=x[mask0], y=y[mask0],
    colorscale=[[0, "rgba(0,0,0,0)"], [1, "blue"]],
    ncontours=15,
    contours=dict(coloring="lines", showlabels=False),
    line=dict(width=1.5),
    showscale=False,
    name="Label 0",
))
fig_lines.add_trace(go.Histogram2dContour(
    x=x[mask1], y=y[mask1],
    colorscale=[[0, "rgba(0,0,0,0)"], [1, "red"]],
    ncontours=15,
    contours=dict(coloring="lines", showlabels=False),
    line=dict(width=1.5),
    showscale=False,
    name="Label 1",
))
fig_lines.update_layout(title="Curve di livello - Label 0 (blu) vs Label 1 (rosso)",
                         xaxis_title="UMAP 1", yaxis_title="UMAP 2",
                         width=800, height=650)
fig_lines.show()
fig_lines.write_html("umap_contours_only.html")

UMAP( verbose=True)
Tue Jul  7 14:43:44 2026 Construct fuzzy simplicial set
Tue Jul  7 14:43:44 2026 Finding Nearest Neighbors
Tue Jul  7 14:43:44 2026 Building RP forest with 17 trees
Tue Jul  7 14:43:49 2026 NN descent for 16 iterations
	 1  /  16
	 2  /  16
	 3  /  16
	 4  /  16
	 5  /  16
	Stopping threshold met -- exiting after 5 iterations
Tue Jul  7 14:44:08 2026 Finished Nearest Neighbor Search
Tue Jul  7 14:44:14 2026 Construct embedding


Epochs completed:   0%|            0/200 [00:00]

	completed  0  /  200 epochs
	completed  20  /  200 epochs
	completed  40  /  200 epochs
	completed  60  /  200 epochs
	completed  80  /  200 epochs
	completed  100  /  200 epochs
	completed  120  /  200 epochs
	completed  140  /  200 epochs
	completed  160  /  200 epochs
	completed  180  /  200 epochs
Tue Jul  7 14:44:41 2026 Finished embedding


In [8]:
# Cella 3 - apri html salvato
from IPython.display import IFrame
IFrame(src="umap_contours_only.html", width=950, height=750)


In [6]:
IFrame(src="umap_density_label0.html", width=950, height=750)


In [7]:
IFrame(src="umap_density_label1.html", width=950, height=750)

In [15]:
# Cella 4 - trova picco piu' alto, prendi 10 punti vicini a esso
import numpy as np
from scipy import ndimage
from scipy.spatial import cKDTree

def trova_top_picco_e_vicini(xs, ys, ids, bins=60, n_vicini=50):
    H, xedges, yedges = np.histogram2d(xs, ys, bins=bins)

    # trova solo il picco massimo assoluto
    row, col = np.unravel_index(np.argmax(H), H.shape)
    val = H[row, col]

    xc = (xedges[row] + xedges[row + 1]) / 2
    yc = (yedges[col] + yedges[col + 1]) / 2

    # trova i 10 punti reali piu' vicini al centro del picco
    tree = cKDTree(np.column_stack([xs, ys]))
    _, nearest_idxs = tree.query([[xc, yc]], k=n_vicini)
    nearest_idxs = nearest_idxs[0]

    ids_vicini = [int(ids[i]) for i in nearest_idxs]
    return {"picco_x": float(xc), "picco_y": float(yc), "densita": float(val),
            "idx_vicini": ids_vicini}

top_label0 = trova_top_picco_e_vicini(x[mask0], y[mask0], idx_sample[mask0])
top_label1 = trova_top_picco_e_vicini(x[mask1], y[mask1], idx_sample[mask1])

print("=== Picco top Label 0 ===")
print(top_label0)
print("\n=== Picco top Label 1 ===")
print(top_label1)

=== Picco top Label 0 ===
{'picco_x': 9.424406051635742, 'picco_y': 1.7644091844558716, 'densita': 133.0, 'idx_vicini': [433642, 315962, 13385, 546923, 550000, 388999, 466644, 153535, 50230, 552432, 151645, 446730, 535096, 20054, 530179, 377743, 325286, 363552, 143182, 477867, 136092, 8974, 573677, 73871, 337040, 150636, 37000, 453458, 167663, 114993, 187721, 499396, 14596, 174510, 316744, 93316, 464950, 155805, 219167, 320178, 151193, 66289, 394826, 233046, 207898, 57543, 574982, 17431, 320495, 460956]}

=== Picco top Label 1 ===
{'picco_x': 6.397908687591553, 'picco_y': 3.889272451400757, 'densita': 171.0, 'idx_vicini': [403277, 171656, 263940, 502939, 564868, 315652, 93329, 49765, 165178, 320213, 241926, 96654, 478512, 386398, 396935, 385287, 119757, 444330, 573022, 425907, 321547, 345495, 483106, 552569, 484230, 408942, 344747, 283508, 173755, 68195, 550721, 326815, 138954, 539831, 62653, 336603, 217657, 228646, 526959, 329566, 139381, 256202, 433139, 230194, 157387, 512512, 228147

In [ ]:
# Cella 5 - estrai frasi corrispondenti agli ID dei picchi
# Nota: ordine assunto identico a R-obi/ai-text-detection (stesso conteggio: 577k righe)
!pip install -q datasets

from datasets import load_dataset

ds = load_dataset("srikanthgali/ai-text-detection-pile-cleaned", split="train")


In [21]:
def stampa_frasi(idx_list, nome_label):
    print(f"\n=== Frasi vicine al picco - {nome_label} ===")
    for i in idx_list:
        entry = ds[i]
        print(f"\nidx {i}")
        print(f"label reale (generated): {entry['generated']}")
        print(f"testo: {entry['text'][:400]}...")
 
stampa_frasi(top_label0["idx_vicini"], "Label 0")
stampa_frasi(top_label1["idx_vicini"], "Label 1")


=== Frasi vicine al picco - Label 0 ===

idx 433642
label reale (generated): 0
testo: Identification of Opportunities and Threats  Integrated SWOT: Soft Drink Indusry Report Table of Contents 1. Introduction 2. Strengths 3. Weakness 4. Opportunities and Threats Introduction The soft drink industry is among the most profitable industries in the market. This is based on the fact that the market demand for soft drink is very relatively high and the suppliers for the product are few. S...

idx 315962
label reale (generated): 0
testo: Relationship Value in Business Markets Essay Critical Writing Abstract The realm of marketing is slowly shifting its focus from the former product centric paradigm to a base that is taking care of the value needs of the customers. The major aim of the relationship marketing is maintaining the relationship between the business and the company to its immediate micro environment. These include the su...

idx 13385
label reale (generated): 0
testo: CharmSoju Post

In [18]:
# Cella - carica ParaDetect (DeBERTa-v3-large + LoRA) e testa una frase
!pip install -q -U torchao peft transformers

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel
import torch

REPO = "srikanthgali/paradetect-deberta-v3-lora"

tokenizer = AutoTokenizer.from_pretrained(REPO)
base_model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/deberta-v3-large", num_labels=2
)
model = PeftModel.from_pretrained(base_model, REPO)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

def predict_text_origin(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True,
                        max_length=512, padding=True).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)
        pred = torch.argmax(probs, dim=-1).item()
    return {
        "prediction": "AI" if pred == 1 else "Human",
        "human_probability": probs[0][0].item(),
        "ai_probability": probs[0][1].item(),
    }



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 34.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 52.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 25.2 MB/s eta 0:00:00


Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-large
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight       

adapter_model.safetensors:   0%|          | 0.00/115M [00:00<?, ?B/s]

In [19]:
# --- prova la tua frase ---
frase_test = "I don't want to talk about this in a political context but I have to say that this is a good way to look at the whole thing. The US is a rich country and rich people love to be rich, but they are not happy with it. So, they have to get rid of that. I feel like that should be the case. But, if you look at how people like Donald Trump are, that's not going to happen. But, the fact is that they are not happy with the way people are treated here and they are going to be very unhappy with it. I'm looking at a lot of things and I think it's important for people to understand the situation and have a conversation with the people. We have to understand what they do, how they feel, what they do not like about it. It's important for people to see that they are not doing this because they feel that there is no way to change it. Donald Trump can do this, but only if you are very careful and you are very passionate about it. If you are not. So, I think it's important for people to understand that if you are not going to change things, the only option is to change things. There was no discussion around the fact that we are a rich country and rich people love to be rich. I think that's a lot of things that we don't want to change but it's a fact that we are rich. We are rich. We can do this. In this context, the question is, how can you change that? This is very important because it is our country. It is our nation. We can do this. In the US, people are not happy with the way things are treated here. But the fact is that we are a rich country. If you look at this picture, it is not about people being poor or being white, it is about the way that we treat them and how we treat other people. You have a country that is rich and rich people love to be rich. It's not about what happens"
print(predict_text_origin(frase_test))

{'prediction': 'AI', 'human_probability': 0.06585693359375, 'ai_probability': 0.93408203125}


In [20]:
frasi_test = [
    """Clinton talks about her time of 'reflection' during sick days  Hillary Clinton returned to the campaign trail Thursday afternoon, debuting some new intro music and telling the crowd that her sick days allowed her a chance to "reconnect with what this whole campaign is about."  The former secretary of state, who took the stage to James Brown's "I Feel Good," spent the beginning of the week at her home in Chappaqua, New York, after being diagnosed late last week with pneumonia.""",
    """Her campaign initially did not disclose the illness and only did so after Clinton was forced to leave an event early on Sunday commemorating the 15th anniversary of the Sept.""",
    """And I'm not great at taking it easy, even under ordinary circumstances, but with just two months to go until Election Day, sitting at home was pretty much the last place I wanted to be."  "But it turns out having a few days to myself was actually a gift, I talked with some old friends.""",
    """The campaign trail doesn't really encourage reflection.""",
    """And it's important to sit with your thoughts every now and then and that did help me reconnect with what this whole campaign is about."  Clinton compared her own ability to take a handful of sick days to that of many Americans who she said are forced to "either go to work sick or they lose a paycheck." She said those Americans, and others "living on a razor's edge" with an aging parent who needs help or without the means to afford a college education, are the reason she is running for president.""",
    """Until that happens NAIAS is not an optimal venue for driving our mission to put more EVs on the road."  The decision is in-line with comments made by Diarmuid O'Connell, Tesla vice president of business development, on Nov.""",
    """We've always had a very good experience at Detroit," a company spokesperson said in a statement.""",
    """Hence he confined his revolutionary discovery to one line in a popular guide to plant identification rather than publishing in some academic journal.""",
    """Richard Evan Schultes was no drug crazed hippy, but a respected Harvard professor with impeccable credentials.""",
    """And if anyone wonders what is the cost of discrimination are, just ask the people and businesses of North Carolina, look at what's happening with the NCAA and the ACC."""
]

for i, f in enumerate(frasi_test, 1):
    print(i, predict_text_origin(f))

1 {'prediction': 'Human', 'human_probability': 0.8642578125, 'ai_probability': 0.1358642578125}
2 {'prediction': 'Human', 'human_probability': 0.88720703125, 'ai_probability': 0.11260986328125}
3 {'prediction': 'Human', 'human_probability': 0.91357421875, 'ai_probability': 0.0863037109375}
4 {'prediction': 'Human', 'human_probability': 0.7333984375, 'ai_probability': 0.266845703125}
5 {'prediction': 'Human', 'human_probability': 0.88623046875, 'ai_probability': 0.1136474609375}
6 {'prediction': 'Human', 'human_probability': 0.833984375, 'ai_probability': 0.165771484375}
7 {'prediction': 'AI', 'human_probability': 0.216552734375, 'ai_probability': 0.783203125}
8 {'prediction': 'Human', 'human_probability': 0.974609375, 'ai_probability': 0.025390625}
9 {'prediction': 'Human', 'human_probability': 0.8828125, 'ai_probability': 0.116943359375}
10 {'prediction': 'AI', 'human_probability': 0.39306640625, 'ai_probability': 0.60693359375}
